# 02 — EDA and RFM

This notebook is a *view* over the modules in `src/analysis/`, not a second implementation.
Anything worth keeping moves into a module so it can be tested and re-run.

Set `DATA_SOURCE=csv` in `.env` to run without MySQL.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config, transform
from src.analysis import revenue, rfm, delivery
from src.utils import md_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

In [ ]:
fact, items, distance = transform.load_all()
print("fact rows      :", f"{len(fact):,}")
print("revenue orders :", f"{int(fact['is_revenue_order'].sum()):,}")
print("delivered      :", f"{int(fact['is_delivered'].sum()):,}")
fact.head(3)

## BQ1 — revenue and growth

Read GMV together with orders. If GMV grows while orders are flat, growth is coming from price
and mix, not from more customers — and that has a ceiling.

In [ ]:
monthly = revenue.monthly_kpis(fact)
md_table(monthly[["purchase_month", "orders", "gmv", "aov", "gmv_mom_pct", "gmv_3mo_ma"]])

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly["purchase_month"], monthly["gmv"] / 1000, color="#7c9dc4", label="GMV")
ax.plot(monthly["purchase_month"], monthly["gmv_3mo_ma"] / 1000, color="#12395c", linewidth=2.4,
        label="3-month moving average")
ax.set_ylabel("GMV (R$ thousands)")
ax.tick_params(axis="x", rotation=90)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.show()

## BQ2 — who is the customer base?

Two rules: grain is `customer_unique_id`, and recency is scored descending so a recent buyer
gets a 5. Getting either backwards produces a plausible-looking table that is completely wrong.

In [ ]:
table = rfm.build_rfm(fact)
summary = rfm.segment_summary(table)
md_table(summary[["segment", "customers", "pct_of_customers", "pct_of_gmv", "avg_orders"]])

In [ ]:
# The trap, visible: correct basis vs the per-order id
rfm.repeat_rate(fact)

`customer_id` is per order, so grouping by it makes every customer a one-timer and retention reads
0%. This table exists so that mistake can never be made silently.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
s = summary.sort_values("gmv")
ax.barh(s["segment"], s["pct_of_gmv"], color="#4b7ba8", label="% of GMV")
ax.barh(s["segment"], s["pct_of_customers"], height=0.35, color="#c9d8e8", label="% of customers")
ax.set_xlabel("% of total")
ax.legend(frameon=False)
plt.show()

## BQ3 — delivery

Late rate and median delivery days travel together. A low late rate with slow deliveries just means
the promise was set too loose.

In [ ]:
md_table(delivery.headline(fact))

In [ ]:
md_table(delivery.by_state(fact).head(10))

## Next

`python -m src.pipeline --stage analyze` runs all six analysis modules and writes every table to
`reports/tables/`, then `make report` assembles `reports/results_pack.md`.